In [ ]:
from pathlib import Path
import os, sys

# Auto-set repo root (expects Data/ and predictions/ at root)
cwd = Path.cwd()
REPO_ROOT = None
for parent in [cwd, *cwd.parents]:
    if (parent / "Data").exists() and (parent / "predictions").exists():
        REPO_ROOT = parent
        break
if REPO_ROOT is None:
    REPO_ROOT = cwd
os.chdir(REPO_ROOT)

# Add notebook-local modules to import path
src_candidates = [
    REPO_ROOT / "notebooks",
    REPO_ROOT / "code_repository" / "notebooks",
]
for SRC in src_candidates:
    if SRC.exists() and str(SRC) not in sys.path:
        sys.path.insert(0, str(SRC))
        break
print(f"Working dir: {Path.cwd()}")


In [ ]:
from analysis.plot_style import apply_style, save_figure, set_save_dir
FIGURE_ID = "supplementary"
FIG_DIR = set_save_dir(REPO_ROOT, FIGURE_ID)
apply_style()


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PRED_ROOT = Path("predictions/251028_tabm_LOO/binary_loo_ablation/sentence_BioBERT_tabm_Text_Embedding+Proteomics+ECFP+CellPaint+MIC+GC")
files = sorted(PRED_ROOT.glob("fold*_rep*_seed*.pkl"))

if not files:
    raise FileNotFoundError(f"No fold*_rep*_seed*.pkl files found in {PRED_ROOT}")


def load_pairwise_pkls(paths):
    dfs = []
    for p in paths:
        df = pd.read_pickle(p)
        if "RowID" not in df.columns:
            df = df.reset_index().rename(columns={"index": "RowID"})
        else:
            df = df.reset_index(drop=True)

        need = ["CompoundID", "CandidateMoA", "ProbaPos", "True", "Pred", "seed", "SampleID", "RowID"]
        df = df[[c for c in need if c in df.columns]]
        dfs.append(df)

    out = pd.concat(dfs, axis=0, ignore_index=True)
    out["CompoundID"] = out["CompoundID"].astype(str)
    out["CandidateMoA"] = out["CandidateMoA"].astype(str)
    out["ProbaPos"] = pd.to_numeric(out["ProbaPos"], errors="coerce")
    out["True"] = pd.to_numeric(out["True"], errors="coerce").fillna(0).astype(int)
    return out


df_all = load_pairwise_pkls(files)

true_pairs = (
    df_all[df_all["True"] == 1][["CompoundID", "CandidateMoA"]]
    .drop_duplicates()
    .groupby("CompoundID")["CandidateMoA"]
    .apply(lambda s: "; ".join(sorted(set(s))))
    .rename("TrueMoA")
)

per_pair = (
    df_all.groupby(["CompoundID", "CandidateMoA"], as_index=False)["ProbaPos"]
    .mean()
    .rename(columns={"ProbaPos": "Score"})
)

scores_wide = per_pair.pivot(index="CompoundID", columns="CandidateMoA", values="Score")

cols = scores_wide.columns.to_numpy()
mat = scores_wide.to_numpy()
mat_f = np.where(np.isnan(mat), -np.inf, mat)

top1 = mat_f.argmax(axis=1)
pred_lbl = cols[top1]
pred_scr = mat[np.arange(mat.shape[0]), top1]

order = np.argsort(-mat_f, axis=1)
top2 = np.where(mat_f.shape[1] >= 2, order[:, 1], order[:, 0])
second_lbl = cols[top2]
second_scr = mat[np.arange(mat.shape[0]), top2]

summary = (
    pd.DataFrame({
        "CompoundID": scores_wide.index.astype(str),
        "TrueMoA": true_pairs.reindex(scores_wide.index).values,
        "PredictedMoA": pred_lbl.astype(str),
        "PredScore": pred_scr.astype(float),
        "SecondMoA": second_lbl.astype(str),
        "SecondScore": second_scr.astype(float),
        "Margin": (pred_scr - second_scr).astype(float),
    })
    .sort_values("CompoundID")
    .reset_index(drop=True)
)

summary

out_path = Path("string_analysis/loo_tabm_all_features_predictions.csv")
summary.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


In [ ]:
summary

In [ ]:
rows = summary["TrueMoA"] != summary["PredictedMoA"]


In [ ]:
summary[rows]

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PRED_ROOT = Path("predictions/251028_tabm_LOO/binary_loo_ablation/sentence_BioBERT_tabm_Text_Embedding+Proteomics+ECFP+CellPaint+MIC+GC")
TXT_DIR = Path("string_analysis")


def stem_to_compound(stem):
    m = re.search(r"jh(\d+)", stem, flags=re.IGNORECASE)
    return f"JH{m.group(1)}" if m else None


def comp_sort_key(cid):
    m = re.search(r"(\d+)", str(cid))
    return int(m.group(1)) if m else 9999


proteomics_compounds = sorted({
    stem_to_compound(p.stem)
    for p in TXT_DIR.glob("jh*.txt")
    if stem_to_compound(p.stem)
}, key=comp_sort_key)

files = sorted(PRED_ROOT.glob("fold*_rep*_seed*.pkl"))
if not files:
    raise FileNotFoundError(f"No fold*_rep*_seed*.pkl files found in {PRED_ROOT}")


def load_pairwise_pkls(paths):
    dfs = []
    for p in paths:
        df = pd.read_pickle(p)
        if "RowID" not in df.columns:
            df = df.reset_index().rename(columns={"index": "RowID"})
        keep = ["CompoundID", "CandidateMoA", "ProbaPos"]
        df = df[keep]
        dfs.append(df)
    out = pd.concat(dfs, axis=0, ignore_index=True)
    out["CompoundID"] = out["CompoundID"].astype(str)
    out["CandidateMoA"] = out["CandidateMoA"].astype(str)
    out["ProbaPos"] = pd.to_numeric(out["ProbaPos"], errors="coerce")
    return out


df_all = load_pairwise_pkls(files)

df_all = df_all[df_all["CompoundID"].isin(proteomics_compounds)].copy()

per_pair = (
    df_all.groupby(["CompoundID", "CandidateMoA"], as_index=False)["ProbaPos"].mean()
)
comp_wide = per_pair.pivot(index="CompoundID", columns="CandidateMoA", values="ProbaPos")
comp_wide = comp_wide.reindex(proteomics_compounds)


def topk_by_row(wide_df: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    rows = []
    for cid, row in wide_df.iterrows():
        vals = row.dropna()
        if vals.empty:
            continue
        topk = vals.sort_values(ascending=False).head(k)
        for rank, (moa, prob) in enumerate(topk.items()):
            rows.append({"CompoundID": str(cid), "MoA": str(moa), "Prob": float(prob), "Rank": rank})
    return pd.DataFrame(rows)


top3 = topk_by_row(comp_wide, k=3)
if top3.empty:
    raise RuntimeError("Top-3 extraction produced no rows (check PKL contents).")

# --- Canonicalization + fixed colors (from 251103_special_samples_analysis.ipynb) ---
COLORS = ["#264653", "#287271", "#2a9d8f", "#8ab17d",
          "#e9c46a", "#efb366", "#f4a261", "#ee8959", "#e76f51"]

DISPLAY_ORDER = [
    "Fatty acid", "Nitrofurans", "Folic acid", "Membrane",
    "RNA synth.", "LPS synth.", "DNA repl.", "Cell wall", "Protein syn."
]

CANON_MAP = {
    "Fatty acid":      ["Fatty acid", "Fatty acid synthesis"],
    "Nitrofurans":     ["Nitrofurans"],
    "Folic acid":      ["Folic acid", "Folic acid metabolism", "Folic acid biosynthesis"],
    "Membrane":        ["Membrane", "Membrane disruption", "Membrane integrity"],
    "RNA synth.":       ["RNA synth.", "RNA Synthesis", "RNA synthesis"],
    "LPS synth.":       ["LPS synth.", "lipopolysaccharide synthesis", "LPS synthesis"],
    "DNA repl.":        ["DNA repl.", "DNA replication", "DNA Replication"],
    "Cell wall":       ["Cell wall", "Cell wall biosynthesis", "Cell-wall biosynthesis"],
    "Protein syn.":     ["Protein syn.", "Protein synthesis", "Protein Synthesis"],
}


def _norm(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.replace("-", " ")
    s = s.replace(".", "")
    s = " ".join(s.split())
    return s.strip().lower()


ALIAS_TO_CANON = {}
for canon, aliases in CANON_MAP.items():
    for a in aliases:
        ALIAS_TO_CANON[_norm(a)] = canon


def canonicalize_moa(s: str) -> str:
    return ALIAS_TO_CANON.get(_norm(s), s)


MOA_TO_COLOR = {m: c for m, c in zip(DISPLAY_ORDER, COLORS)}
FALLBACK_COLOR = "#9aa0a6"

# Canonicalize + colors

top3["MoA_canon"] = top3["MoA"].astype(str).map(canonicalize_moa)
top3["CompoundID"] = pd.Categorical(top3["CompoundID"], categories=proteomics_compounds, ordered=True)
top3 = top3.sort_values(["CompoundID", "Rank"], ascending=[True, True])
bar_colors = top3["MoA_canon"].map(MOA_TO_COLOR).fillna(FALLBACK_COLOR)

# X positions (grouped by compound)
x_centers = np.arange(len(proteomics_compounds))
k = 3
bar_width = 0.24
offsets = {r: (r - (k - 1) / 2) * bar_width for r in range(k)}

x_pos = top3.apply(
    lambda r: x_centers[proteomics_compounds.index(r["CompoundID"])] + offsets[int(r["Rank"])],
    axis=1
)

plt.figure(figsize=(max(8, len(proteomics_compounds) * 0.6), 5), dpi=120)
plt.bar(
    x_pos,
    top3["Prob"].values,
    width=bar_width * 0.95,
    color=bar_colors.values,
    edgecolor="black",
    linewidth=0.4,
)

plt.ylabel("Probability (per MoA)")
plt.xlabel("Compound")
plt.title("Top-3 predicted MoAs per compound (proteomics subset)")
plt.xticks(x_centers, proteomics_compounds, rotation=45, ha="right")
plt.ylim(0, 1.1)

moas_present = list(dict.fromkeys(top3["MoA_canon"].astype(str)))
legend_order = [m for m in DISPLAY_ORDER if m in set(moas_present)]
handles = [Patch(facecolor=MOA_TO_COLOR[m], edgecolor="black", label=m) for m in legend_order]
plt.legend(handles=handles, title="MoA", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

plt.tight_layout()
save_figure(plt.gcf(), TXT_DIR / "loo_tabm_proteomics_top3_moas_barplot.pdf", dpi=300, format="pdf")
plt.show()
